In [0]:
!pip install pyomo

In [0]:
import torch

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch.nn as nn

from sklearn.preprocessing import StandardScaler, MinMaxScaler

# Set seed for reproducibility
np.random.seed(42)
CREATE_DATA = False
LOAD_CASE_STUDY = False
LOAD_SIMPLE_DATA = True 

In [0]:

def scale_columns_standard(df, columns_to_scale):
    scaler = StandardScaler()
    df[columns_to_scale] = scaler.fit_transform(df[columns_to_scale])
    return df

def scale_columns_minmax(df, columns_to_scale):
    scaler = MinMaxScaler()
    df[columns_to_scale] = scaler.fit_transform(df[columns_to_scale])
    return df

'''def generate_simple_synthetic_data(n_samples=1000, seed=42):
    np.random.seed(seed)

    # 2 input features, randomly sampled
    x1 = np.random.uniform(0, 1, size=n_samples)
    x2 = np.random.uniform(0, 1, size=n_samples)

    # Define true mean and std as simple linear functions
    true_mean = 2 * x1 + 3 * x2       # range: [0, 5]
    true_std = 0.1 + 0.5 * np.abs(x1 - x2)  # range: [0.1, 0.6]

    # Sample y from N(mean, std)
    y = np.random.normal(loc=true_mean, scale=true_std)

    # Return as dataframe + y
    X = pd.DataFrame({'x1': x1, 'x2': x2})
    y = pd.Series(y, name='y')
    
    return X, y, true_mean, true_std'''

def generate_simple_synthetic_data(n_samples=1000, seed=42):
    np.random.seed(seed)

    # Contextual features
    x1 = np.random.uniform(0, 1, size=n_samples)   # e.g., income level normalized [0, 1]
    x2 = np.random.uniform(0, 1, size=n_samples)   # e.g., brand loyalty normalized [0, 1]

    # Price feature (higher price usually decreases demand)
    price = np.random.uniform(1, 10, size=n_samples)  # price between $1 and $10

    # Define true mean (nonlinear relationship)
    true_mean = (5 * x1 + 3 * x2) / (1 + 0.5 * price)  # demand decreases with higher price

    # Define true std (slightly depends on features)
    true_std = 0.2 + 0.3 * np.abs(x1 - x2) + 0.05 * (price / 10)  # more price variation adds more noise

    # Sample y from N(mean, std)
    y = np.random.normal(loc=true_mean, scale=true_std)

    # Package into dataframe
    X = pd.DataFrame({'x1': x1, 'x2': x2, 'price': price})
    y = pd.Series(y, name='y')

    return X, y, true_mean, true_std

'''X, y, true_mean, true_std = generate_simple_synthetic_data()
print(X.head())
print(y.head())'''

def create_synthetic_csv(num_clients=100, num_transactions=1000, num_products=3, client_categories=["low_spender", "mid_spender", "high_spender"]):

    # Generate client-level data
    client_ids = np.arange(1, num_clients + 1)

    # Income: Should follow a right skewed distribution (log-normal)
    # Adjusting the income distribution to follow a log-normal distribution (right-skewed, long tail)
    mean_income = 10  # Log-normal mean
    sigma_income = 0.6  # Log-normal standard deviation

    # Generate income values (skewed distribution)
    client_incomes = np.random.lognormal(mean=mean_income, sigma=sigma_income, size=num_clients)
    client_incomes = np.interp(client_incomes, (client_incomes.min(), client_incomes.max()), (20000, 200000))

    #client_incomes = np.random.uniform(20000, 100000, size=num_clients)  # Continuous feature: income
    client_spending_categories = np.random.choice(client_categories, size=num_clients)  # Categorical feature

    # Map spending categories to a numerical effect (low = 0.8, mid = 1, high = 1.2)
    category_effects = {"low_spender": 0.8, "mid_spender": 1, "high_spender": 1.2}
    product_base_prices = {1: (10, 30), 2: (15, 50), 3: (5, 25)}

    data = []
    for _ in range(num_transactions):
        client = np.random.choice(client_ids)
        product = np.random.randint(1, num_products + 1)

        # Determine product base price range
        base_price_min, base_price_max = product_base_prices[product]
        mean_price = (base_price_min + base_price_max) / 2

        # Client attributes
        income = client_incomes[client - 1]
        category = client_spending_categories[client - 1]
        category_multiplier = category_effects[category]

        # Price is sampled randomly within the product's base range
        price = np.random.uniform(base_price_min, base_price_max)

        # Amount bought depends on price (inverse relation) and income (higher income buys more)
        base_demand = np.random.uniform(5, 20)  # Base demand per product
        income_factor = (income / 200000)  # Scaled income effect (higher income = more purchases)
        
        '''print("&"*50)
        print("price: ", price)
        print("mean_price: ", mean_price)
        print("price / mean_price: ", price / mean_price)
        print("(price / mean_price)**2: ", (price / mean_price)**2)
        print("(1 / (1 + (price / mean_price)**2)): ", (1 / (1 + (price / mean_price)**2)))
        print("&"*50)'''
        #amount_bought = max(0, (base_demand * category_multiplier * income_factor * (1 / (1 + (price / mean_price)**2))) + np.random.normal(0, 1))
        #amount_bought = max(0, (base_demand * category_multiplier * income_factor * (1 / (price / mean_price)**2)) + np.random.normal(0, 1))

        # Separate deterministic and stochastic parts
        deterministic_part = (base_demand * category_multiplier * income_factor * (1 / (price / mean_price)**2))
        #print("deterministic_part: ", deterministic_part)
        # Stochastic part: noise based on price and income. Each row has a different noise. amount_boughtᵢ ~ Normal(μᵢ, σᵢ²)
        std_dev = 0.5 + 0.1 * (price / mean_price)**2 + 0.1 * (1 - income_factor)
        #print("std_dev: ", std_dev)
        noise = np.random.normal(0, std_dev)
        print("noise: ", noise)
        amount_bought = max(0, (deterministic_part + noise))

        data.append([client, product, price, amount_bought, income, category])

    # Create DataFrame
    df = pd.DataFrame(data, columns=["client_id", "product_id", "offered_price", "amount_bought", "income", "client_category"])
    df["amount_bought"] = df["amount_bought"].round().astype(int)
    df["amount_bought"] = df["amount_bought"].clip(lower=0)  # Ensure no negative values

    df_name_to_save = f"synthetic_dataset_{num_clients}clients_{num_transactions}trans.csv"
    df.to_csv("data/" + df_name_to_save, index=False)
    return df, df_name_to_save

def load_case_study_data():
    data = pd.read_csv('../DistCL_code/data_house_japan/data_clean.csv', index_col=0)
    data.index = pd.to_datetime(data.index)

    y = data['demand']
    X = data.drop(['demand'], axis=1, inplace=False)

    val_ind = X.index[(X.index >= pd.to_datetime('2018-11-01')) & (X.index < pd.to_datetime('2018-12-01'))]
    test_ind = X.index[X.index >= pd.to_datetime('2018-12-01')]

    return X, y, val_ind, test_ind


In [0]:
if CREATE_DATA:
    print("Creating synthetic data...")
    df, df_name = create_synthetic_csv(num_clients=1, num_transactions=10000, num_products=3, client_categories=["low_spender", "mid_spender", "high_spender"])

In [0]:
if LOAD_CASE_STUDY:
    print("Loading case study data...")
    X, y, val_ind, test_ind = load_case_study_data()
    display(X.head())

elif LOAD_SIMPLE_DATA:
    X, y, true_mean, true_std = generate_simple_synthetic_data(n_samples=1000)
    
    scale_columns_minmax(X, ['x1', 'x2'])
    print(X.head())
    print(y.head())
else:
    print("Loading synthetic data...")
    # Load the synthetic dataset
    df = pd.read_csv("data/synthetic_dataset_1clients_10000trans.csv")
    df = df[df['client_id'] == 1]  # Filter for client_id = 1
    display(df.head())
    print("len(df):", len(df))

    df = df[df['product_id'] ==3 ]

    plt.figure(figsize=(4, 4))
    sns.scatterplot(x=df["offered_price"], y=df["amount_bought"], alpha=0.5)
    plt.xlabel("Offered Price")
    plt.ylabel("Amount Bought")
    plt.title("Amount Bought vs Offered Price")
    plt.show()

    target = 'amount_bought'
    #numerical_features = ['offered_price', 'income']
    numerical_features = ['offered_price']
    categorical_features = ['client_category']
    columns_to_use = numerical_features + [target]
    correlation_matrix = df[numerical_features + [target]].corr()
    print(correlation_matrix)

    scale_columns_minmax(df, numerical_features)


In [0]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

model_config = {
    'hidden_layers': 3,
    'hidden_size': 64,
    'drop': 0.2
    }

training_config = {
    'lr': 0.01,
    'wd': 0.0001,
    'batchsize': 64,
    'iters': 1000,
    'patience': 200,
    }



In [0]:
'''trainer = Trainer(X, y, model_config, training_config)
trainer.create_data_partitions()
print("X_train shape: ", trainer.X_train.shape)
print("X_val shape: ", trainer.X_val.shape)
print("X_test shape: ", trainer.X_test.shape)
trainer.train()
trainer.test()'''

In [0]:
from trainer import Trainer

In [0]:
np.random.seed(42)
torch.manual_seed(1)
if LOAD_CASE_STUDY:
    print("Loading case study data...")
    trainer = Trainer(X, y, model_config, training_config, device=device)
    trainer.set_data_partitions(X_train=X[(~X.index.isin(val_ind)) & (~X.index.isin(test_ind))], y_train=y[(~y.index.isin(val_ind)) & (~y.index.isin(test_ind))], X_val=X[X.index.isin(val_ind)], y_val=y[y.index.isin(val_ind)], X_test=X[X.index.isin(test_ind)], y_test=y[y.index.isin(test_ind)])
elif LOAD_SIMPLE_DATA:
    print("Loading simple synthetic data...")
    trainer = Trainer(X, y, model_config, training_config, device=device)
    trainer.create_data_partitions(val_size=0.2, test_size=0.2, random_state=42)
else:
    print("Loading synthetic data...")
    trainer = Trainer(df[['offered_price', 'income']], df['amount_bought'], model_config=model_config, training_config=training_config, device=device)

    trainer.create_data_partitions(val_size=0.2, test_size=0.2, random_state=42)

print("X_train shape: ", trainer.X_train.shape)
print("X_val shape: ", trainer.X_val.shape)
print("X_test shape: ", trainer.X_test.shape)

trainer.train()

trainer.test()

In [0]:
trainer.evaluate_gaussian_nll()

In [0]:
trainer.evaluate_standardized_residuals()

In [0]:
trainer.model

In [0]:
y_train_tensor = torch.tensor(trainer.y_train.values, dtype=torch.float32).unsqueeze(1)  # shape [N, 1]

baseline_mean = torch.mean(y_train_tensor)
baseline_std = torch.std(y_train_tensor) + 1e-6

# Repeat the values for every row
baseline_mean_preds = baseline_mean.repeat(y_train_tensor.shape[0], 1)
baseline_std_preds = baseline_std.repeat(y_train_tensor.shape[0], 1)

# Square std to get variance
baseline_var_preds = baseline_std_preds ** 2

criterion = nn.GaussianNLLLoss()
baseline_loss = criterion(baseline_mean_preds, y_train_tensor, baseline_var_preds)

print(f"Baseline Gaussian NLL Loss: {baseline_loss.item():.4f}")

pred_mean, pred_sd = trainer.predict(trainer.X_val, return_tensor=True)     # X_val: torch tensor of shape [N, D]
pred_var = pred_sd ** 2               # square to get variance
y_val_tensor = torch.tensor(trainer.y_val.values, dtype=torch.float32).unsqueeze(1).to(device)
model_loss = criterion(pred_mean, y_val_tensor, pred_var)

print(f"Model Gaussian NLL Loss: {model_loss.item():.4f}")

In [0]:
pred_sd

In [0]:
def plot_calibration(y_true, y_pred, y_std):
    """
    Creates a histogram of standardized residuals (Z-scores) to check uncertainty calibration.
    
    Parameters:
    - y_true: True target values (numpy array).
    - y_pred: Predicted mean (numpy array).
    - y_std: Predicted standard deviation (numpy array).
    
    Returns:
    - Calibration plot
    """
    z_scores = np.abs(y_true - y_pred) / y_std  # Standardized residuals

    plt.figure(figsize=(8, 5))
    plt.hist(z_scores, bins=30, density=True, alpha=0.6, color='g', label='Standardized Residuals')

    # Reference lines at 1σ and 2σ
    plt.axvline(1, color='r', linestyle='dashed', linewidth=2, label='1 Std Dev')
    plt.axvline(2, color='b', linestyle='dashed', linewidth=2, label='2 Std Dev')

    plt.xlabel('Z-Score (|y_true - y_pred| / std)')
    plt.ylabel('Density')
    plt.legend()
    plt.title('Calibration Plot: Standardized Residuals')
    plt.show()

plot_calibration(trainer.y_test, mean_pred, sd_pred)
